In [0]:
# crear base de datos para silver
spark.sql("CREATE DATABASE IF NOT EXISTS capa_silver")
spark.sql("USE capa_silver")


DataFrame[]

In [0]:
# Tratamiento de feriados

from pyspark.sql import functions as F

fer_bronze = spark.read.format("delta").load(
    spark.sql("DESCRIBE DETAIL capa_bronze.feriados_delta").first().location
)

fer_silver = (fer_bronze
    # normalización de nombres
    .withColumnRenamed("fecha",  "holiday_date_raw")
    .withColumnRenamed("nombre", "name_raw")
    .withColumnRenamed("tipo",   "type_raw")
    # tipado
    .withColumn("holiday_date", F.to_date("holiday_date_raw", "yyyy-MM-dd"))
    .withColumn("name",  F.trim(F.col("name_raw")))
    .withColumn("type",  F.trim(F.col("type_raw")))
    # calidad: fecha no nula, nombre no nulo
    .filter(F.col("holiday_date").isNotNull() & F.col("name").isNotNull())
    # deduplicado por fecha+nombre
    .dropDuplicates(["holiday_date","name"])
    .select("holiday_date","name","type")
)

silver_feriados_path = spark.sql("DESCRIBE DETAIL capa_bronze.feriados_delta").first().location \
    .replace("/bronze/", "/silver/").rsplit("/",1)[0].replace("delta","feriados_delta")

(fer_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .save(silver_feriados_path))

spark.sql(f"""
CREATE TABLE IF NOT EXISTS capa_silver.feriados
USING delta
LOCATION '{silver_feriados_path}'
""")

# chequeo rápido
display(spark.table("capa_silver.feriados").orderBy("holiday_date").limit(10))


holiday_date,name,type
2024-01-01,Año nuevo,inamovible
2024-02-12,Carnaval,inamovible
2024-02-13,Carnaval,inamovible
2024-03-24,Día Nacional de la Memoria por la Verdad y la Justicia,inamovible
2024-03-29,Viernes Santo,inamovible
2024-04-01,Feriado puente turístico,puente
2024-04-02,Día del Veterano y de los Caídos en la Guerra de Malvinas,inamovible
2024-05-01,Día del Trabajador,inamovible
2024-05-25,Día de la Revolución de Mayo,inamovible
2024-06-17,Paso a la Inmortalidad del General Martín Güemes,trasladable


In [0]:
%sql
select * from capa_bronze.vuelos_delta limit 10

IATAdestorig,acft_body,acftype,aerolinea,arpt,atda,belt,blockoff,blockon,checkins,chk_from,chk_lyf,chk_to,color,destorig,estbr,estes,estin,etda,gate,id,id_flight_reg,id_flight_tp,id_flight_tra,idaerolinea,idclimaicono,idshared,logo,matricula,mov,nro,pasajeros,posicion,rot,sdphrase,sdtemp,sdtempunit,sector,stda,term,termsec,tipoVuelo,via,ingest_date
AEP,NB,null,AEROLINEAS ARGENTINAS,CTC,,,,,004-006,null,null,null,#C0C0C0,Aeroparque,No Horario,En Horario,On Time,,1,7652552,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1455,,001,AR 1454,null,null,null,P,23/12 11:00,null,P,null,,2024-12-23
AEP,NB,null,AEROLINEAS ARGENTINAS,CTC,,,,,004-006,null,null,null,#C0C0C0,Aeroparque,No Horario,En Horario,On Time,,1,7652553,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1458,,001,AR 1458,null,null,null,P,23/12 18:55,null,P,null,IRJ,2024-12-23
AEP,NB,null,AEROLINEAS ARGENTINAS,CRD,,,,,001-004,null,null,null,#C0C0C0,Aeroparque,No Horario,En Horario,On Time,,03,7652548,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1837,,02,AR 1836,null,null,null,P,23/12 01:50,null,P,null,,2024-12-23
AEP,NB,null,AEROLINEAS ARGENTINAS,CRD,,,,,001-004,null,null,null,#C0C0C0,Aeroparque,No Horario,En Horario,On Time,,03,7652545,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1823,,01,AR 1822,null,null,null,P,23/12 08:10,null,P,null,,2024-12-23
USH,NB,null,LADE,CRD,,,,,005-006,null,null,null,#C0C0C0,Ushuaia,No Horario,En Horario,On Time,,01,7652536,C,1,P,5U,null,,https://www.aeropuertosargentina.com/img/aerolineas/5U_200.GIF,,D,5U 444,,03A,,null,null,null,P,23/12 08:30,null,P,null,"PMQ,FTE,RGL",2024-12-23
NQN,NB,null,AEROLINEAS ARGENTINAS,CRD,,,,,001-004,null,null,null,#C0C0C0,Neuquén,No Horario,En Horario,On Time,,03,7652543,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1431,,02,AR 1824,null,null,null,P,23/12 11:15,null,P,null,,2024-12-23
COR,NB,null,AEROLINEAS ARGENTINAS,CRD,,,,,001-004,null,null,null,#C0C0C0,Córdoba,No Horario,En Horario,On Time,,03,7652544,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1559,,01,AR 1828,null,null,null,P,23/12 13:00,null,P,null,,2024-12-23
AEP,NB,null,FLYBONDI,CRD,,,,,007-008,null,null,null,#C0C0C0,Aeroparque,No Horario,En Horario,On Time,,04,7652550,C,1,P,FO,null,,https://www.aeropuertosargentina.com/img/aerolineas/FO_200.GIF,,D,FO 5501,,01,FO 5500,null,null,null,P,23/12 14:45,null,P,null,,2024-12-23
AEP,NB,null,AEROLINEAS ARGENTINAS,CRD,,,,,001-004,null,null,null,#C0C0C0,Aeroparque,No Horario,En Horario,On Time,,03,7652546,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1829,,02,AR 1558,null,null,null,P,23/12 16:20,null,P,null,,2024-12-23
AEP,NB,null,AEROLINEAS ARGENTINAS,CRD,,,,,001-004,null,null,null,#C0C0C0,Aeroparque,No Horario,En Horario,On Time,,03,7652547,C,1,P,AR,null,,https://www.aeropuertosargentina.com/img/aerolineas/AR_200.GIF,,D,AR 1835,,01,AR 1834,null,null,null,P,23/12 20:55,null,P,null,,2024-12-23


In [0]:
# Tratamiento de vuelos

from pyspark.sql import functions as F

# Cargar Bronze (Delta)
v_bronze_path = spark.sql("DESCRIBE DETAIL capa_bronze.vuelos_delta").first().location
v = spark.read.format("delta").load(v_bronze_path)

# Usamos la fecha
yyyymmdd = F.date_format(F.col("ingest_date"), "yyyy-MM-dd")
year_from_path = F.date_format(F.col("ingest_date"), "yyyy")

def to_ts(col):
    dd  = F.regexp_extract(F.col(col), r"^(\d{2})/(\d{2}) (\d{2}):(\d{2})$", 1)
    MM  = F.regexp_extract(F.col(col), r"^(\d{2})/(\d{2}) (\d{2}):(\d{2})$", 2)
    hh  = F.regexp_extract(F.col(col), r"^(\d{2})/(\d{2}) (\d{2}):(\d{2})$", 3)
    mi  = F.regexp_extract(F.col(col), r"^(\d{2})/(\d{2}) (\d{2}):(\d{2})$", 4)
    return F.to_timestamp(F.concat_ws(" ",
                F.concat_ws("-", year_from_path, MM, dd),
                F.concat_ws(":", hh, mi)
           ), "yyyy-MM-dd HH:mm")

sched_ts = to_ts("stda")   # horario programado
est_ts   = to_ts("etda")   # estimado (si viene)
act_ts   = to_ts("atda")   # real (si viene)

# Origen/Destino a partir de mov: 'D' (sale) / 'A' (llega)
origin = F.when(F.col("mov")=="D", F.col("arpt")).otherwise(F.col("IATAdestorig"))
dest   = F.when(F.col("mov")=="D", F.col("IATAdestorig")).otherwise(F.col("arpt"))

# Normalización de aerolínea
air_norm_map = {
  "AEROLÍNEAS ARGENTINAS":"Aerolíneas Argentinas",
  "AR":"Aerolíneas Argentinas",
  "JETSMART":"JetSmart",
  "FLYBONDI":"Flybondi"
}
from pyspark.sql.functions import create_map, lit
air_mapping = create_map([lit(x) for kv in air_norm_map.items() for x in kv])

v_silver = (v
  .withColumn("flight_date", F.to_date(yyyymmdd))  # yyyy-MM-dd
  .withColumn("sched_ts", sched_ts)
  .withColumn("est_ts",   est_ts)
  .withColumn("act_ts",   act_ts)
  .withColumn("origin", origin)
  .withColumn("destination", dest)
  .withColumn("airline", F.coalesce(air_mapping[F.upper(F.col("aerolinea"))], F.col("aerolinea")))
  .withColumn("airline_code", F.col("idaerolinea"))
  .withColumn("flight_number", F.col("nro"))
  # demora (en minutos) si hay act_ts
  .withColumn("delay_minutes",
      F.when(F.col("act_ts").isNotNull(),
             (F.col("act_ts").cast("long") - F.col("sched_ts").cast("long"))/60.0)
  )
  # calidad: IATA de 3 chars, fechas presentes, aerolínea no vacía
  .filter(F.length("origin")==3)
  .filter(F.length("destination")==3)
  .filter(F.col("flight_date").isNotNull())
  .filter(F.col("airline").isNotNull())
  .select(
      "flight_date","sched_ts","est_ts","act_ts",
      "origin","destination",
      "airline","airline_code","flight_number",
      "mov","estes","estin","estbr",
      "delay_minutes"
  )
)

silver_vuelos_path = v_bronze_path.replace("/bronze/", "/silver/").rsplit("/",1)[0].replace("delta","vuelos_delta")

(v_silver.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema","true")
  .save(silver_vuelos_path))

spark.sql(f"""
CREATE TABLE IF NOT EXISTS capa_silver.vuelos
USING delta
LOCATION '{silver_vuelos_path}'
""")

# chequeo rápido
display(spark.table("capa_silver.vuelos").limit(20))


flight_date,sched_ts,est_ts,act_ts,origin,destination,airline,airline_code,flight_number,mov,estes,estin,estbr,delay_minutes
2024-12-23,2024-12-23T11:00:00Z,null,null,CTC,AEP,AEROLINEAS ARGENTINAS,AR,AR 1455,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T18:55:00Z,null,null,CTC,AEP,AEROLINEAS ARGENTINAS,AR,AR 1458,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T01:50:00Z,null,null,CRD,AEP,AEROLINEAS ARGENTINAS,AR,AR 1837,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T08:10:00Z,null,null,CRD,AEP,AEROLINEAS ARGENTINAS,AR,AR 1823,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T08:30:00Z,null,null,CRD,USH,LADE,5U,5U 444,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T11:15:00Z,null,null,CRD,NQN,AEROLINEAS ARGENTINAS,AR,AR 1431,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T13:00:00Z,null,null,CRD,COR,AEROLINEAS ARGENTINAS,AR,AR 1559,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T14:45:00Z,null,null,CRD,AEP,Flybondi,FO,FO 5501,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T16:20:00Z,null,null,CRD,AEP,AEROLINEAS ARGENTINAS,AR,AR 1829,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T20:55:00Z,null,null,CRD,AEP,AEROLINEAS ARGENTINAS,AR,AR 1835,D,En Horario,On Time,No Horario,null


In [0]:
%sql
select * from capa_silver.vuelos limit 10

flight_date,sched_ts,est_ts,act_ts,origin,destination,airline,airline_code,flight_number,mov,estes,estin,estbr,delay_minutes
2024-12-23,2024-12-23T11:00:00Z,null,null,CTC,AEP,AEROLINEAS ARGENTINAS,AR,AR 1455,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T18:55:00Z,null,null,CTC,AEP,AEROLINEAS ARGENTINAS,AR,AR 1458,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T01:50:00Z,null,null,CRD,AEP,AEROLINEAS ARGENTINAS,AR,AR 1837,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T08:10:00Z,null,null,CRD,AEP,AEROLINEAS ARGENTINAS,AR,AR 1823,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T08:30:00Z,null,null,CRD,USH,LADE,5U,5U 444,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T11:15:00Z,null,null,CRD,NQN,AEROLINEAS ARGENTINAS,AR,AR 1431,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T13:00:00Z,null,null,CRD,COR,AEROLINEAS ARGENTINAS,AR,AR 1559,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T14:45:00Z,null,null,CRD,AEP,Flybondi,FO,FO 5501,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T16:20:00Z,null,null,CRD,AEP,AEROLINEAS ARGENTINAS,AR,AR 1829,D,En Horario,On Time,No Horario,null
2024-12-23,2024-12-23T20:55:00Z,null,null,CRD,AEP,AEROLINEAS ARGENTINAS,AR,AR 1835,D,En Horario,On Time,No Horario,null
